In [1]:
from pathlib import Path
import importlib
import functions
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
importlib.reload(functions)
from tqdm.notebook import tqdm
import math
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, roc_auc_score,
    precision_recall_curve, average_precision_score, accuracy_score
)
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.dummy import DummyClassifier

In [2]:
PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "datasets/Dummy"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
#whole dataset
dataset = functions.GeneExpressionDataset(
    exp_matrix_path= PROJECT_ROOT / 'datasets/exp_mat_unscaled.csv',
    conf_matrix_path= PROJECT_ROOT / 'datasets/adj_giant.csv',
    threshold=0.1
)

k = 10
n = 9
split_seed = 1905
y = np.array([data.y.item() for data in dataset])
country = np.array(dataset.country)
strat = np.array([f"{label}__{group}" for label, group in zip(y, country)])
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=split_seed)

C:\Users\fabia\Desktop\Git_Repos\aap-transcriptomic-analysis\functions.py:174: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  edge_index = torch.tensor([src, dst], dtype=torch.long)


In [4]:
D = np.array([data.y.item() for data in dataset])

folds: [np.ndarray] = []
for _, fold_idx in skf.split(np.zeros(len(D)), strat):
    folds.append(fold_idx)

metrics_t_strat = {
    "Accuracy": np.array([]),
    "F1": np.array([]),
    "Precision": np.array([]),
    "Recall": np.array([]),
    "AUC-PRC": np.array([]),
    "AUC-ROC": np.array([])
}

metrics_t_freq = {
    "Accuracy": np.array([]),
    "F1": np.array([]),
    "Precision": np.array([]),
    "Recall": np.array([]),
    "AUC-PRC": np.array([]),
    "AUC-ROC": np.array([])
}

total = k
pbar = tqdm(total=total, desc="Verarbeitung läuft")

for i in range(k):
    test_idx = folds[i]

    other_fold_ids = [m for m in range(k) if m != i] 
    train_idx_full = np.concatenate([folds[r] for r in range(k) if r != i], axis=0)

    pd.DataFrame(train_idx_full, columns=["index"]).to_csv(OUTPUT_DIR / f"fold_{i}_train_full.csv", index=False)
    pd.DataFrame(test_idx, columns=["index"]).to_csv(OUTPUT_DIR / f"fold_{i}_test.csv", index=False)
    functions.removeBatchEffect(OUTPUT_DIR / f"fold_{i}_train_full.csv",
                      OUTPUT_DIR / f"fold_{i}_test.csv",
                      str(i), "Dummy")

    train_dataset = functions.GeneExpressionDataset(
        exp_matrix_path= OUTPUT_DIR / f"train_corrected_{i}.csv",
        conf_matrix_path= PROJECT_ROOT / "datasets" /'adj_giant.csv',
        threshold=0.1
    )
    test_dataset = functions.GeneExpressionDataset(
        exp_matrix_path= OUTPUT_DIR / f"test_corrected_{i}.csv",
        conf_matrix_path= PROJECT_ROOT / "datasets" /'adj_giant.csv',
        threshold=0.1
    )
    
    scaler = StandardScaler()
    scaler.fit(train_dataset.expr)

    X_train = scaler.transform(train_dataset.expr)
    y_train = train_dataset.labels
    X_test = scaler.transform(test_dataset.expr)
    y_test = test_dataset.labels

    model_strat = DummyClassifier(strategy="stratified", random_state=2026)
    model_strat.fit(X_train, y_train)

    y_pred_strat = model_strat.predict(X_test)
    y_prob_strat = model_strat.predict_proba(X_test)[:, 1] 
    
    model_freq = DummyClassifier(strategy="most_frequent", random_state=2026)
    model_freq.fit(X_train, y_train)

    y_pred_freq = model_freq.predict(X_test)
    y_prob_freq = model_freq.predict_proba(X_test)[:, 1] 

    metrics_t_strat['Precision'] = np.append(metrics_t_strat['Precision'], precision_score(y_test, y_pred_strat, average="macro", zero_division=0))
    metrics_t_strat['Recall'] = np.append(metrics_t_strat['Recall'], recall_score(y_test, y_pred_strat, average="macro", zero_division=0))
    metrics_t_strat['Accuracy'] = np.append(metrics_t_strat['Accuracy'], accuracy_score(y_test, y_pred_strat))
    metrics_t_strat['F1'] = np.append(metrics_t_strat['F1'], f1_score(y_test, y_pred_strat, average="macro", zero_division=0))
    metrics_t_strat['AUC-PRC'] = np.append(metrics_t_strat['AUC-PRC'], average_precision_score(y_test, y_prob_strat))
    metrics_t_strat['AUC-ROC'] = np.append(metrics_t_strat['AUC-ROC'], roc_auc_score(y_test, y_prob_strat))

    metrics_t_freq['Precision'] = np.append(metrics_t_freq['Precision'], precision_score(y_test, y_pred_freq, average="macro", zero_division=0))
    metrics_t_freq['Recall'] = np.append(metrics_t_freq['Recall'], recall_score(y_test, y_pred_freq, average="macro", zero_division=0))
    metrics_t_freq['Accuracy'] = np.append(metrics_t_freq['Accuracy'], accuracy_score(y_test, y_pred_freq))
    metrics_t_freq['F1'] = np.append(metrics_t_freq['F1'], f1_score(y_test, y_pred_freq, average="macro", zero_division=0))
    metrics_t_freq['AUC-PRC'] = np.append(metrics_t_freq['AUC-PRC'], average_precision_score(y_test, y_prob_freq))
    metrics_t_freq['AUC-ROC'] = np.append(metrics_t_freq['AUC-ROC'], roc_auc_score(y_test, y_prob_freq))
    pbar.update(1)
    
pbar.close()
    

Verarbeitung läuft:   0%|          | 0/10 [00:00<?, ?it/s]

In [5]:
metrics_t_strat

{'Accuracy': array([0.52380952, 0.61904762, 0.54761905, 0.54761905, 0.51219512,
        0.51219512, 0.53658537, 0.56097561, 0.65853659, 0.48780488]),
 'F1': array([0.44297082, 0.55437666, 0.48148148, 0.48148148, 0.43681319,
        0.43681319, 0.47542088, 0.46982759, 0.58764368, 0.39578947]),
 'Precision': array([0.44166667, 0.55833333, 0.48333333, 0.48333333, 0.43534483,
        0.43534483, 0.47701149, 0.46982759, 0.58764368, 0.39367816]),
 'Recall': array([0.44642857, 0.55357143, 0.48518519, 0.48518519, 0.44047619,
        0.44047619, 0.47948718, 0.46982759, 0.58764368, 0.39835165]),
 'AUC-PRC': array([0.31547619, 0.36309524, 0.35079365, 0.35079365, 0.32186411,
        0.32186411, 0.35718157, 0.2820122 , 0.34434282, 0.29393371]),
 'AUC-ROC': array([0.44642857, 0.55357143, 0.48518519, 0.48518519, 0.44047619,
        0.44047619, 0.47948718, 0.46982759, 0.58764368, 0.39835165])}

In [6]:
metrics_t_freq

{'Accuracy': array([0.66666667, 0.66666667, 0.64285714, 0.64285714, 0.65853659,
        0.65853659, 0.63414634, 0.70731707, 0.70731707, 0.68292683]),
 'F1': array([0.4       , 0.4       , 0.39130435, 0.39130435, 0.39705882,
        0.39705882, 0.3880597 , 0.41428571, 0.41428571, 0.4057971 ]),
 'Precision': array([0.33333333, 0.33333333, 0.32142857, 0.32142857, 0.32926829,
        0.32926829, 0.31707317, 0.35365854, 0.35365854, 0.34146341]),
 'Recall': array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]),
 'AUC-PRC': array([0.33333333, 0.33333333, 0.35714286, 0.35714286, 0.34146341,
        0.34146341, 0.36585366, 0.29268293, 0.29268293, 0.31707317]),
 'AUC-ROC': array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])}

In [7]:
import numpy as np

def latex_table_from_results(results, model_name="Baseline", digits=2):
    metrics = list(results.keys())
    
    row = []
    for m in metrics:
        values = np.array(results[m])
        mean = values.mean()
        std = values.std(ddof=1)  # Stichproben-Std
        row.append(f"{mean:.{digits}f} $\\pm$ {std:.{digits}f}")
    
    latex = []
    latex.append("\\begin{table}[h]")
    latex.append("\\centering")
    latex.append("\\begin{tabular}{l" + "c"*len(metrics) + "}")
    latex.append("\\hline")
    latex.append("Model & " + " & ".join(metrics) + " \\\\")
    latex.append("\\hline")
    latex.append(model_name + " & " + " & ".join(row) + " \\\\")
    latex.append("\\hline")
    latex.append("\\end{tabular}")
    latex.append("\\caption{Performance (mean $\\pm$ std)}")
    latex.append("\\end{table}")
    
    return "\n".join(latex)

In [8]:
print(latex_table_from_results(metrics_t_strat))

\begin{table}[h]
\centering
\begin{tabular}{lcccccc}
\hline
Model & Accuracy & F1 & Precision & Recall & AUC-PRC & AUC-ROC \\
\hline
Baseline & 0.55 $\pm$ 0.05 & 0.48 $\pm$ 0.06 & 0.48 $\pm$ 0.06 & 0.48 $\pm$ 0.06 & 0.33 $\pm$ 0.03 & 0.48 $\pm$ 0.06 \\
\hline
\end{tabular}
\caption{Performance (mean $\pm$ std)}
\end{table}


In [9]:
print(latex_table_from_results(metrics_t_freq))

\begin{table}[h]
\centering
\begin{tabular}{lcccccc}
\hline
Model & Accuracy & F1 & Precision & Recall & AUC-PRC & AUC-ROC \\
\hline
Baseline & 0.67 $\pm$ 0.03 & 0.40 $\pm$ 0.01 & 0.33 $\pm$ 0.01 & 0.50 $\pm$ 0.00 & 0.33 $\pm$ 0.03 & 0.50 $\pm$ 0.00 \\
\hline
\end{tabular}
\caption{Performance (mean $\pm$ std)}
\end{table}
